# LLM Extraction — Phi-3-mini (offline)
Extracts 5 high-value SDRF columns from paper text using Phi-3-mini running fully offline.
Runs 1 inference call per test PXD (15 total). Takes ~10 min on Kaggle T4.

**Columns extracted:** `Characteristics[Modification]`, `Characteristics[Disease]`,
`Characteristics[OrganismPart]`, `Characteristics[CellType]`, `Characteristics[DevelopmentalStage]`

**Prerequisite:** Attach your `phi3-mini-weights` dataset to this notebook.

## 1. Setup — imports and paths

In [1]:
import json, re, warnings
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────
ROOT        = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd().parent
MODEL_ID    = 'Qwen/Qwen2.5-1.5B-Instruct'   # ~3 GB download, fits in 6 GB VRAM
SUB_PATH    = ROOT / 'outputs' / 'submission_with_fallback.csv'
TEST_PATH   = ROOT / 'outputs' / 'test_preprocessed.csv'
OUT_PATH    = ROOT / 'outputs' / 'submission_llm.csv'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'Model  : {MODEL_ID}')
print(f'SUB    : {SUB_PATH}  exists={SUB_PATH.exists()}')

c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device : cuda
GPU    : NVIDIA GeForce GTX 1060
VRAM   : 6.4 GB
Model  : Qwen/Qwen2.5-1.5B-Instruct
SUB    : c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_with_fallback.csv  exists=True


## 2. Load Phi-3-mini

In [2]:
print('Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

print('Loading model (downloads ~3 GB on first run, cached after that) ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16 if device == 'cuda' else torch.float32,
    device_map='auto',
    trust_remote_code=True,
)

pipe = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    temperature=None,
    top_p=None,
    return_full_text=False,   # return only the generated reply, not the prompt
)
print('Model ready.')

Loading tokenizer ...
Loading model (downloads ~3 GB on first run, cached after that) ...


Loading weights: 100%|██████████| 338/338 [00:05<00:00, 58.55it/s]
The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p', 'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model ready.


## 3. Extraction prompt
One prompt per PXD. We ask the model to return JSON with exactly the 5 fields we need.
The prompt includes the canonical SDRF format so the model outputs strings that match
what Kaggle expects.

In [3]:
SYSTEM_PROMPT = """You are an expert proteomics bioinformatician.
Given the text of a scientific publication, extract SDRF metadata fields.
Return ONLY a JSON object with exactly these keys. No explanation, no markdown.

Rules:
- Characteristics[Modification]: list ALL post-translational modifications in SDRF format.
  Each modification: NT=<name>;AC=UNIMOD:<id>;TA=<residue>;MT=<Fixed|Variable>
  Common examples:
    Carbamidomethylation of C → NT=Carbamidomethyl;AC=UNIMOD:4;TA=C;MT=Fixed
    Oxidation of M → NT=Oxidation;AC=UNIMOD:21;TA=M;MT=Variable
    Phosphorylation → NT=Phospho;AC=UNIMOD:21;TA=S;MT=Variable
  If multiple mods, return as array. If none mentioned, return ["Not Applicable"].

- Characteristics[Disease]: disease or condition studied.
  Use controlled vocabulary: e.g. 'colorectal cancer', 'Alzheimer disease', 'normal'.
  If healthy controls only, use 'normal'. If not mentioned, 'Not Applicable'.

- Characteristics[OrganismPart]: tissue/biofluid sampled.
  e.g. 'blood plasma', 'liver', 'brain cortex', 'urine', 'cell culture'.
  If not mentioned, 'Not Applicable'.

- Characteristics[CellType]: specific cell type if mentioned.
  e.g. 'HeLa', 'Jurkat', 'T cell', 'neuron'. If not mentioned, 'Not Applicable'.

- Characteristics[DevelopmentalStage]: life stage of subjects.
  e.g. 'adult', 'embryo', 'child', 'newborn'. If not mentioned, 'Not Applicable'.

Return exactly this JSON structure:
{
  "Characteristics[Modification]": ["NT=...;AC=...;TA=...;MT=..."],
  "Characteristics[Disease]": "disease name or Not Applicable",
  "Characteristics[OrganismPart]": "tissue or Not Applicable",
  "Characteristics[CellType]": "cell type or Not Applicable",
  "Characteristics[DevelopmentalStage]": "stage or Not Applicable"
}"""

def build_prompt(pub_text: str) -> str:
    # Truncate to ~3000 words — methods section is usually in the first half
    words = pub_text.split()
    truncated = ' '.join(words[:3000])
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'Extract SDRF metadata from this publication text:\n\n{truncated}'},
    ]
    # apply_chat_template adds the model-specific tokens and the generation prompt
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

print('Prompt template ready.')

Prompt template ready.


## 4. Parse LLM JSON output

In [4]:
def parse_llm_output(text: str) -> dict:
    """Extract JSON from model output, return empty dict on failure."""
    # Find the JSON block
    match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if not match:
        return {}
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        # Try fixing common issues: trailing commas, single quotes
        cleaned = match.group()
        cleaned = re.sub(r',\s*}', '}', cleaned)
        cleaned = re.sub(r',\s*]', ']', cleaned)
        try:
            return json.loads(cleaned)
        except:
            return {}

LLM_COLS = [
    'Characteristics[Modification]',
    'Characteristics[Disease]',
    'Characteristics[OrganismPart]',
    'Characteristics[CellType]',
    'Characteristics[DevelopmentalStage]',
]

print('Parser ready. LLM target columns:', LLM_COLS)

Parser ready. LLM target columns: ['Characteristics[Modification]', 'Characteristics[Disease]', 'Characteristics[OrganismPart]', 'Characteristics[CellType]', 'Characteristics[DevelopmentalStage]']


## 5. Run inference — one call per PXD

In [5]:
test = pd.read_csv(TEST_PATH, dtype=str)
pxd_texts = test.groupby('PXD')['pub_text'].first().to_dict()
print(f'Test PXDs: {len(pxd_texts)}')

pxd_results: dict[str, dict] = {}

for i, (pxd, text) in enumerate(pxd_texts.items(), 1):
    print(f'[{i:2d}/{len(pxd_texts)}] {pxd} ...', end=' ', flush=True)
    prompt = build_prompt(text)
    try:
        # return_full_text=False means generated_text is already just the reply
        output = pipe(prompt)[0]['generated_text']
        parsed = parse_llm_output(output)
        pxd_results[pxd] = parsed
        mods = parsed.get('Characteristics[Modification]', '?')
        disease = parsed.get('Characteristics[Disease]', '?')
        print(f'disease={disease[:30]}  mods={str(mods)[:40]}')
    except Exception as e:
        print(f'ERROR: {e}')
        pxd_results[pxd] = {}

print(f'\nInference complete. {sum(1 for v in pxd_results.values() if v)} / {len(pxd_results)} succeeded.')

Test PXDs: 15
[ 1/15] PXD004010 ... 

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


disease=Alzheimer's disease  mods=['Not Applicable']
[ 2/15] PXD016436 ... 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


disease=Not Applicable  mods=['Not Applicable']
[ 3/15] PXD019519 ... 

...

disease=Prostate cancer  mods=['Not Applicable']
[14/15] PXD062877 ... 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


disease=Polymerase gamma (Polg)-relate  mods=['Not Applicable']
[15/15] PXD064564 ... 

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


disease=Not Applicable  mods=['Not Applicable']

Inference complete. 15 / 15 succeeded.


## 6. Apply extractions to submission

In [6]:
sub = pd.read_csv(SUB_PATH, dtype=str).fillna('Not Applicable')
sub_out = sub.copy()

# Map PXD to each row
pxd_series = pd.read_csv(TEST_PATH, dtype=str)['PXD'].values

for col in LLM_COLS:
    if col not in sub_out.columns:
        continue

    new_vals = []
    for pxd in pxd_series:
        extracted = pxd_results.get(pxd, {}).get(col, 'Not Applicable')

        # Modification is a list — take first item for base column
        if isinstance(extracted, list):
            val = extracted[0] if extracted else 'Not Applicable'
        else:
            val = str(extracted).strip() if extracted else 'Not Applicable'

        # Don't overwrite a good existing value with Not Applicable
        existing = sub_out.loc[sub_out.index[pxd_series.tolist().index(pxd)
                   if pxd in pxd_series.tolist() else 0], col]
        new_vals.append(val)

    sub_out[col] = new_vals
    filled = (sub_out[col] != 'Not Applicable').sum()
    print(f'{col:<45} {filled:>5} / {len(sub_out)} filled')

# Handle Modification.1, .2 etc with additional mods from the list
for pxd in pxd_results:
    mods = pxd_results[pxd].get('Characteristics[Modification]', [])
    if isinstance(mods, list) and len(mods) > 1:
        mask = pxd_series == pxd
        for j, mod in enumerate(mods[1:], 1):
            col_name = f'Characteristics[Modification].{j}'
            if col_name in sub_out.columns:
                sub_out.loc[mask, col_name] = mod

sub_out.to_csv(OUT_PATH, index=False)
print(f'\nSaved → {OUT_PATH}')
print(f'Shape : {sub_out.shape}')

Characteristics[Modification]                     0 / 1659 filled
Characteristics[Disease]                       1554 / 1659 filled
Characteristics[OrganismPart]                  1538 / 1659 filled
Characteristics[CellType]                      1444 / 1659 filled
Characteristics[DevelopmentalStage]            1390 / 1659 filled

Saved → c:\Users\Sunny\OneDrive\Documents\Kaggle-Harmonizing-the-data-of-your-data\outputs\submission_llm.csv
Shape : (1659, 81)


## 7. Sanity check before uploading

In [7]:
sub_check   = pd.read_csv(OUT_PATH, dtype=str)
_sample_local  = ROOT / 'data' / 'SampleSubmission.csv'
_sample_kaggle = Path('/kaggle/input/harmonizing-the-data-of-your-data/SampleSubmission.csv')
sample = pd.read_csv(_sample_local if _sample_local.exists() else _sample_kaggle, dtype=str)

print(f'Rows match: {len(sub_check) == len(sample)}  ({len(sub_check)} rows)')
print(f'ID match  : {(sub_check["ID"].values == sample["ID"].values).all()}')

print('\nLLM column fill rates:')
for col in LLM_COLS:
    if col in sub_check.columns:
        n = (sub_check[col] != 'Not Applicable').sum()
        pct = n / len(sub_check) * 100
        print(f'  {col:<45} {n:>5} ({pct:.1f}%)')

print('\nSample predictions:')
display(sub_check[['ID','PXD'] + LLM_COLS].head(5))

print('\nUpload outputs/submission_llm.csv to Kaggle.')

Rows match: True  (1659 rows)
ID match  : True

LLM column fill rates:
  Characteristics[Modification]                     0 (0.0%)
  Characteristics[Disease]                       1554 (93.7%)
  Characteristics[OrganismPart]                  1538 (92.7%)
  Characteristics[CellType]                      1444 (87.0%)
  Characteristics[DevelopmentalStage]            1390 (83.8%)

Sample predictions:


,ID,PXD,Characteristics[Modification],Characteristics[Disease],Characteristics[OrganismPart],Characteristics[CellType],Characteristics[DevelopmentalStage]
0,1,PXD004010,Not Applicable,Alzheimer's disease,postmortem brain,Not Applicable,Not Applicable
1,2,PXD004010,Not Applicable,Alzheimer's disease,postmortem brain,Not Applicable,Not Applicable
2,3,PXD004010,Not Applicable,Alzheimer's disease,postmortem brain,Not Applicable,Not Applicable
3,4,PXD004010,Not Applicable,Alzheimer's disease,postmortem brain,Not Applicable,Not Applicable
4,5,PXD004010,Not Applicable,Alzheimer's disease,postmortem brain,Not Applicable,Not Applicable



Upload outputs/submission_llm.csv to Kaggle.
